# Small LLM and prompting using CoT

If you have the overhead for a tiny local model (like Phi-3 or Llama-3-8B), you can use "Chain of Thought" prompting in a single pass. This is the least "lightweight" in terms of compute, but the most "lightweight" in terms of code and setup.

The Prompt Strategy:

"Review: [Text] Attributes Present: [List from your 1st pipeline] Global Score: [Score from your 2nd pipeline] Task: For each attribute, provide a satisfaction score from 1-9. Use the Global Score as a baseline but adjust based on specific adjectives used for that attribute."

In [1]:
import torch
from transformers import pipeline
import pandas as pd
import numpy as np
import os
import json
import time

/users/eleves-a/2022/adrien.bindel/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/users/eleves-a/2022/adrien.bindel/.local/lib/python3.9/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-02-09 16:23:25.930876: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-09 16:23:25.961460: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To e

In [2]:
!pip install accelerate

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
import os
import torch
from transformers import pipeline
from dotenv import load_dotenv

# 1. Load the variables from .env into the environment
load_dotenv()

load_dotenv()
print(f"Token loaded: {os.getenv('HF_TOKEN') is not None}")

def import_pipe(model_name):
    pipe = pipeline(
        "text-generation",
        model=model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",  # Changed "cuda" to "auto" (better practice with accelerate)
        token=os.getenv("HF_TOKEN")
        model_kwargs={"cache_dir": "./llama_model"}
    )

    if pipe.tokenizer.pad_token is None:
        pipe.tokenizer.pad_token = pipe.tokenizer.eos_token

    return pipe

model_name = "meta-llama/Llama-3.2-3B"
pipe = import_pipe(model_name)

# get the user prompt
prompt = "Review: [Text] Attributes Present: [List from your 1st pipeline] Global Score: [Score from your 2nd pipeline] Task: For each attribute, provide a satisfaction score from 1-9. Use the Global Score as a baseline but adjust based on specific adjectives used for that attribute."

# Getting the response from the LLM
start_time = time.time()

messages = [
    {"role": "system", "content": "You are a helpful assistant that analyzes reviews."},
    {"role": "user", "content": "Review: The food was great but the service was slow. Attributes: Food, Service. Global Score: 7. Task: Provide satisfaction scores for each attribute."}
]

outputs = pipe(
    messages,
    max_new_tokens=256,
)
end_time = time.time()
inference_time = end_time - start_time

response = outputs[0]["generated_text"][-1]["content"]

Token loaded: False


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:55<00:00, 27.53s/it]
Device set to use cuda:0


NameError: name 'messages' is not defined

In [ ]:
import torch
import time
from transformers import pipeline

def get_response(model_name, user_prompt):
    # 1. Initialize the pipeline
    pipe = pipeline(
        "text-generation",
        model=model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto", # "auto" is safer for multi-GPU or balanced loading
    )

    # 2. Fix padding (Standard for Llama models)
    if pipe.tokenizer.pad_token is None:
        pipe.tokenizer.pad_token = pipe.tokenizer.eos_token

    # 3. Format the chat message
    messages = [
        {"role": "system", "content": "You are a helpful assistant that scores product attributes."},
        {"role": "user", "content": user_prompt},
    ]

    # 4. Run inference
    start_time = time.time()
    
    outputs = pipe(
        messages,
        max_new_tokens=256,
        do_sample=True,    # Added for more natural scoring logic
        temperature=0.7,   # Controls randomness
    )
    
    end_time = time.time()
    
    # 5. Extract and return
    response = outputs[0]["generated_text"][-1]["content"]
    inference_time = end_time - start_time
    
    return response, inference_time

# --- Usage ---
model_id = "meta-llama/Llama-3.2-3B"
my_prompt = "Review: The battery is great but the screen is dim. Attributes Present: Battery, Screen. Global Score: 7. Task: Provide satisfaction scores 1-9."

res, duration = get_response(model_id, my_prompt)
print(f"Time: {duration:.2f}s\nResponse: {res}")